# Knowledge-Based Robotic Decision-Making
Welcome to the third Chapter of our hands-on course!
Today, you will focus on understanding how a knowledge base supports robotic decision-making. You’ll learn to query the knowledge base to identify necessary actions for tasks, like how to perceive the milk inside the fridge.
### Goal
By the end of the session, you will have successfully made queries to the knowledge base, enabling the robot to determine the steps required to complete its tasks.


## Step 1: Initialization

Import the necessary modules and define the objects in the environment, as we did in Day 2.

In [ ]:
from pycram.ros.tf_broadcaster import TFBroadcaster
from pycram.ros.viz_marker_publisher import VizMarkerPublisher
from pycram.worlds.bullet_world import BulletWorld
from pycram.designators.action_designator import *
from pycram.designators.location_designator import *
from pycram.designators.object_designator import *
from pycram.datastructures.enums import ObjectType, WorldMode, TorsoState
from pycram.datastructures.pose import Pose
from pycram.process_module import simulated_robot
from pycram.object_descriptors.urdf import ObjectDescription
from pycram.world_concepts.world_object import Object
from pycram.datastructures.dataclasses import Color

extension = ObjectDescription.get_file_extension()

world = BulletWorld(WorldMode.DIRECT)
world.allow_publish_debug_poses = True
viz = VizMarkerPublisher()
tf = TFBroadcaster()

robot_name = "pr2"
robot = Object(robot_name, ObjectType.ROBOT, f"{robot_name}{extension}", pose=Pose([1, 2, 0]))

apartment = Object("apartment", ObjectType.ENVIRONMENT, f"apartment-small{extension}")
milk = Object("milk", ObjectType.MILK, "milk.stl", pose=Pose([0.5, 2.5, 1], [0, 0, 0, 1]))
milk.color = Color(0, 0, 1, 1)
milk_desig = BelieveObject(names=["milk"])
robot_desig = BelieveObject(names=[robot_name])
apartment_desig = BelieveObject(names=["apartment"])


### Step 2: Learn how to Query the Knowledge Base

Before we can start quering our knowledge base, we have to establish a connection to it by creating an object for the client:

In [ ]:
knowrob = knowrob_client()

If this has run through successfully (you should see something like "[KnowRbo] done"), we can test the connection by sending a test query written in Prolog:

In [ ]:
knowrob.once("member(X, [1,2,3]).")

Within the PyCram knowrob interface, we differenciate between wanting to receive just one result, or all possible solutions. For this, there are essentially two query functions: **once** and **all_solutions**. In most instances, it is fine to just use **once**. However **all_solutions** might be useful, for building failure handling functionality, in order to iterate over all the possible solutions. To see the difference for the member function, try it out with **all_solutions** instead of **once**. 
Do you notice any other difference between the two functions?

In [ ]:
#Add your code here

<details>

<summary>Click here to get the solution</summary>

```python
knowrob.all_solutions('member(X,[1,2,3]).')
```

The result of **once** is a dictionary, while the result of **all_solutions** is a list of dict. Keep this in mind when working on the results of KnowRob. 
</details>

### Step 3: Learn why we need a Knowledge Base
Now, we would like to go back to our scenario of getting the milk from the fridge. Since the robot doesn't yet know where the milk is located, we could see if maybe knowrob does by posting a query. 
**Important** note: KnowRob uses Prolog as a query language, which allows it to be very powerful, but takes a while to get used to. Think of it as trying to express what you would like to know in a logic formula, and prolog will try and find a solution which matches your query.

The most important thing to know about it is, that everything starting with a capital letter is a variabe which Prolog will fill with knowledge which matches the query, and that you can chain queries with a comma, since that is a logical "and". Below you can find a cheat sheet for Prolog syntax.

<details>

<summary><b> Prolog cheat sheet </b> </summary>


| Symbol          | Meaning                                | Example                                               |
|-----------------|----------------------------------------|-------------------------------------------------------|
| `.`             | End of clause or query                 | `likes(john, pizza).`                                 |
| `,`             | Logical AND                            | `happy(X), healthy(X).`                               |
| `;`             | Logical OR                             | `happy(X); sad(X).`                                   |
| `:-`            | "If" (defines a rule)                  | `happy(X) :- enjoys(X, Y), positive(Y).`              |
| `?-`            | Start a query                          | `?- likes(john, pizza).`                              |
| `=`             | Unification (binds values)             | `X = john.`                                           |
| `is`            | Arithmetic assignment                  | `X is 3 + 4.`                                         |
| `<`, `>`, `=<`, `>=` | Comparison operators        | `X > 5, Y =< 10.`                                     |
| `\=`            | Not equal                              | `X \= john.`                                          |
| `[]`            | Empty list                             | `X = [].`                                             |
| `\|`            | List cons (head and tail separator)    | `[H \| T] = [1, 2, 3].`                               |
| `_`             | Anonymous variable (ignored)           | `likes(_, pizza).`                                    |
| `\+`            | Negation                               | `\+ happy(X).`                                        |
| `!`             | Cut (prevents backtracking)            | `happy(X) :- enjoys(X, Y), !, positive(Y).`           |


</details>

In [ ]:
knowrob.once('entity (an Object(type="Milk", storagePlace="?storagePlace"))')

With this query we say that we are looking for an **entity**, which is an **Object**, which corresponds to the PyCRAM Object Designators. This object should be of **type** milk, and with **?storagePlace** we designate that we would like to know the **storage place** of the object, hence the questionmark. You can think of everything prefixed with a questionmark, as a varibale which gets filled with information by knowrob. 

In [ ]:
# Example code to query the knowledge base
# query which finds out if the fridge is open or not
query = 'entity ( an Object (type="Fridge",location= an Location(handle="?handle" openingState="?open")))'
# Run the query on the knowledge base
result = knowrob.once(query)
print(result)

### Step 4: Plan ahead for the next steps
Now that we know where the milk is located, we can plan the next steps. The robot needs to open the fridge, detect the milk, and grasp it. Let's start by moving the robot to the fridge door. Remember to use the NavigateAction to move the robot to the fridge door. The pose could be: [1.3, 2.5, 0], [0, 0, 1, 0]

In [ ]:
 # Add your code here

<details>

<summary>Click here to get the solution</summary>

```python
nav_pose = Pose([1.3, 2.5, 0], [0, 0, 1, 0])


with simulated_robot:
   NavigateAction(target_locations=[nav_pose]).resolve().perform()  
#If you see the robot moving you can continue with the next cell
```
</details>

The issue now is that the robot might not be able to open the fridge door from its current position. This is because the robot needs to be positioned directly in front of the fridge door and requires sufficient space to open it. Therefore, the robot needs to move to a better position. Manually adjusting its location for every object in the environment would be tedious. Instead, we can leverage the knowledge base to query the location of the fridge door handle and use costmaps to determine an optimal position for opening the door.





### Step 5: Understanding Costmaps


Costmaps are a way for robots to assess their surroundings by assigning numerical values (or "costs") to different areas of the environment based on specific criteria. This helps the robot understand which areas are visible, or preferable for certain tasks. The higher the value of a pose, the more likely it is to be chosen by the robot.

#### Types of Costmaps

 **a. Visibility Costmap**
A visibility costmap determines which poses around a target position can observe the target. This is especially useful for robots with cameras that can change their height, allowing them to adjust their view to detect objects or obstacles more effectively.

**Example Scenario**: If a robot needs to monitor a specific object, it uses a visibility costmap to identify which positions it should move to in order to keep that object in sight.

**b. Occupancy Costmap**
An occupancy costmap marks which areas in the environment are free of obstacles and safe for the robot to navigate. It essentially maps out all the positions where the robot can move without colliding with objects. The parameter "distance_to_obstacle" is used to set the minimum distance between the robot and any obstacle.

**Example Scenario**: When planning a path, the robot uses this map to avoid bumping into furniture or walls.

**c. Semantic Costmap**
A semantic costmap marks an area over a link of an object, that allows to dynamically determine potential poses on the surface of the object. This is useful for tasks where the robot needs to interact with objects or surfaces in a specific way.

**Example Scenario**: If a robot is looking to plae an object, we can use the semantic costmap to find a suitable location dynamically. 

**d. Gaussian Costmap**
A Gaussian costmap assigns values based on the distance from a certain region, with the highest values at the center of that region. The region can be determined by a "distance" parameter, where "distance" denotes the distance of the peak from the center of the costmap. A distance of 0 creates a costmap with a single peak at the center of the costmap. This is useful for tasks where proximity to a specific location is important.

**Example Scenario**: The robot uses this map to decide how close it should be to a person or object to interact effectively.

**e. Directional Costmap**
A directional costmap allows to create costmap that covers exactly half of the area around an object in one direction that may be specified. This is useful for tasks where the robot needs to interact with objects or surfaces in a specific direction.

**Example Scenario**: If an object needs to be interacted with from the front, this costmap can help finding poses that are in front of the object, instead of behind it.

#### Multiplying Costmaps
Robots can multiply different costmaps to factor in multiple criteria simultaneously. For instance, combining visibility and occupancy costmaps allows the robot to find a spot that is both visible and free of obstacles, since poses that cannot observe an object, or are too close to an obstacle, will have a cost of 0, leaving only the overlap of the two costmaps.

**Example Scenario**: If a robot is asked to monitor an area while avoiding collisions, it uses a combined costmap to determine an optimal position that satisfies both conditions.

#### Prioritizing Poses in Costmaps
Robots can prioritize poses of a primary costmap by using a secondary costmap. The overlap of the two costmaps will be high priority poses, while poses in the primary costmap that were not part of the overlap will be lower priority, but not completely ignored.

**Example Scenario**: If a robot needs to place an object on a table, it can use a primary costmap to find possible poses from which to place from, and a directional costmap to determine from which side of the table to approach. Should all prioritized poses fail for some reason, the low priority poses may still be tried out, as as a fallback.

By using costmaps, robots can make informed decisions about where to move or position themselves, enabling them to navigate and interact with their environment more intelligently.

You can find the example costmaps in the picture below:


![Costmap1](img/costmaps.001.jpeg)


## Step 6: Opening Action
Opening allows the robot to open a Container, the container is identified by an ObjectPart designator which describes the handle of the drawer that should be grasped. The OpeningAction needs to know which arm should be used to open the container. The ObjectPart would look like this:  

```python
ObjectPart(names=["frdige_door_link_handle"], part_of=apartment_desig)
```  

It takes the name of the handle as a string and the part_of designator of the apartment. This name is corresponding to the name of the handle in the URDF file. As you can tell naming all the parts in the URDF file is crucial for the robot to be able to interact with them, but knowing them during coding is quite annoying and frustrating. But we can use our knowledge base to ask for specific parts or names of objects.  

**Question**: Do you know what type of joint the fridge door has?
<details>

<summary>Click here to get the solution</summary>

It is a revolute joint.
</details>

In [ ]:
# Placeholder code for querying the knowledge base for the name of the fridge door
query = 'entity (an Object(type="Fridge",location= an Location(handle="?handle" openingState="?open")))'
actions = knowrob.once(query)
for action in actions:
    print(action)

Another side note, how does the robot now actual knows that it has to open fridge, obviously we have to tell it. But how? We can use the knowledge base to ask fo required actions. 

In [ ]:
# Placeholder code for querying the knowledge base for the name of the fridge door
query = 'required_action(robot, perceive, milk).'
actions = knowrob.once(query)
for action in actions:
    print(action)

**Exercise**: Now write the code to open the fridge door, and then detect the milk.
Remember: Use the Costmaps to determine the optimal position for the robot to open the fridge door, query the knowledge base for the fridge door handle, and then open the fridge door. The costmap will help you determine the optimal position for the robot to open the fridge door, the method is called AccessingLocation: 
```python
closed_location, opened_location = AccessingLocation(handle_desig=handle_designator.resolve(),
                                                         robot_desig=robot_desig.resolve()).resolve()
```

The `OpenAction` method allows a robot to open an object like a door or cabinet using its arm. Here's a quick guide on how to use it. 
#### Usage

To open an object, you need:

- An **object designator** that specifies what the robot should interact with (e.g., a handle).
- The **arm** that the robot will use for the action.
- The **start and goal locations** that define the positions before and after opening the object.

```python
OpenAction(
    object_designator_description=_, 
    arms=[closed_location.arms[0]], 
    start_goal_location=[_, opened_location]
).resolve().perform()
```
Now use everything together!

In [ ]:
#Add: your code here

In [ ]:
TODO @VANESSA, @ALINA the code should contain the queries and how to put them into the necessary spots

<details>

<summary>Click here to get the solution</summary>

```python

with simulated_robot:
    start_pose = Pose([1.3, 2.7, 0], [0, 0, 1, 0])
    milk_target_pose = Pose([5.34, 3.55, 0.8])

    NavigateAction([start_pose]).resolve().perform()
    ParkArmsAction([Arms.BOTH]).resolve().perform()
    MoveTorsoAction([TorsoState.HIGH]).resolve().perform()
    
    handle_designator = ObjectPart(names=["handle_cab3_door_top"], part_of=apartment_desig.resolve())
    closed_location, opened_location = AccessingLocation(handle_desig=handle_designator.resolve(),
                                                         robot_desig=robot_desig.resolve()).resolve()
    OpenAction(object_designator_description=handle_designator, arms=[closed_location.arms[0]],
               start_goal_location=[closed_location, opened_location]).resolve().perform()
    LookAtAction(targets=[milk_desig.resolve().pose]).resolve().perform()
    object_designator = DetectAction(milk_desig).resolve().perform()
```
</details>